# 10 — Router Eval: Run Benchmark on LLMs

Loads the benchmark from notebook 09 and runs each item against multiple LLMs via a Router pattern adapted from `C:/Projekty/Claude assistant/multiagent_tool.py`.

**Models tested**:
- `claude-sonnet-4-5` (via Anthropic API)
- `claude-opus-4-1` (via Anthropic API)  
- Optional: `llama3` via Ollama (local)

Each model is asked the **same** entailment question with no logic priming (we want the model's *implicit* logic, not whatever logic we tell it to use).

Output: `benchmark_data/llm_responses.json` — verdict + reasoning trace per (model, item).

In [1]:
import json
import os
import re
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path

# Load .env if present
env_path = os.path.expanduser("~/.claude/.env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip()

import anthropic

# Load benchmark
with open("benchmark_data/llm_logic_benchmark.json", encoding="utf-8") as f:
    benchmark = json.load(f)
print(f"Loaded {len(benchmark)} benchmark items")

Loaded 12 benchmark items


## Prompt design

Critical: the prompt must **not** mention any logic by name. We want the model's *natural* judgment.

We force a structured response:
1. `verdict`: "entails" or "does_not_entail"
2. `confidence`: 1–5
3. `reasoning`: the model's explanation

In [2]:
ENTAILMENT_PROMPT = """You are evaluating whether a conclusion logically follows from premises.

Premises:
{premises_block}

Conclusion: {conclusion}

Question: Does the conclusion logically follow from the premises?

Respond with ONLY a JSON object (no code fences, no commentary):
{{
  "verdict": "entails" or "does_not_entail",
  "confidence": 1-5,
  "reasoning": "one or two sentences explaining your judgment"
}}"""

def make_prompt(item: dict) -> str:
    premises_block = "\n".join(f"  {i+1}. {p}" for i, p in enumerate(item["premises"]))
    return ENTAILMENT_PROMPT.format(premises_block=premises_block, conclusion=item["conclusion"])

# Show one example
print(make_prompt(benchmark[0]))

You are evaluating whether a conclusion logically follows from premises.

Premises:
  1. There exists either a largest twin prime or there does not.

Conclusion: The disjunction 'there is a largest twin prime, or there is not' is true.

Question: Does the conclusion logically follow from the premises?

Respond with ONLY a JSON object (no code fences, no commentary):
{
  "verdict": "entails" or "does_not_entail",
  "confidence": 1-5,
  "reasoning": "one or two sentences explaining your judgment"
}


## Router: dispatch to model backends

Same pattern as `multiagent_tool.py` but simplified: one router function per model.

In [3]:
client = anthropic.Anthropic()

def extract_json(text: str) -> dict:
    """Strip code fences, extract first {...}."""
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        text = m.group(0)
    return json.loads(text)

def query_anthropic(model: str, prompt: str, max_tokens: int = 512) -> dict:
    """Single call to Anthropic API. Returns dict with verdict/confidence/reasoning, or error."""
    try:
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        try:
            return extract_json(text)
        except Exception:
            return {"verdict": "PARSE_ERROR", "confidence": 0, "reasoning": text[:200]}
    except Exception as e:
        return {"verdict": "API_ERROR", "confidence": 0, "reasoning": str(e)[:200]}

def query_ollama(model: str, prompt: str) -> dict:
    """Optional local Ollama backend. Returns same shape."""
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://localhost:11434/api/generate",
            data=json.dumps({"model": model, "prompt": prompt, "stream": False}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=120) as resp:
            text = json.loads(resp.read())["response"]
        try:
            return extract_json(text)
        except Exception:
            return {"verdict": "PARSE_ERROR", "confidence": 0, "reasoning": text[:200]}
    except Exception as e:
        return {"verdict": "API_ERROR", "confidence": 0, "reasoning": str(e)[:200]}

def route(model_id: str, prompt: str) -> dict:
    if model_id.startswith("claude"):
        return query_anthropic(model_id, prompt)
    if model_id.startswith("ollama:"):
        return query_ollama(model_id.split(":", 1)[1], prompt)
    raise ValueError(f"Unknown model backend: {model_id}")

## Run sweep

Each item is asked **N times** per model (default 3) so we can measure within-model consistency. Non-determinism in LLMs reveals whether the "logic" is stable or noisy.

In [4]:
MODELS = [
    "claude-sonnet-4-5",
    # "claude-opus-4-1",       # uncomment if you have access
    # "ollama:llama3",          # uncomment if Ollama is running locally
]

N_TRIALS = 3   # repeat each (model, item) N times
DELAY = 0.3    # seconds between calls (rate limit politeness)

results = []

for model in MODELS:
    print(f"\n=== {model} ===")
    for item in benchmark:
        for trial in range(N_TRIALS):
            prompt = make_prompt(item)
            response = route(model, prompt)
            results.append({
                "model": model,
                "item_id": item["id"],
                "category": item["category"],
                "trial": trial,
                "verdict": response.get("verdict", ""),
                "confidence": response.get("confidence", 0),
                "reasoning": response.get("reasoning", ""),
                "expected": item["expected"],
            })
            print(f"  {item['id']:12s} trial={trial} → {response.get('verdict','?')}")
            time.sleep(DELAY)

with open("benchmark_data/llm_responses.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} responses to benchmark_data/llm_responses.json")


=== claude-sonnet-4-5 ===
  LEM-01       trial=0 → entails
  LEM-01       trial=1 → entails
  LEM-01       trial=2 → entails
  LEM-02       trial=0 → does_not_entail
  LEM-02       trial=1 → does_not_entail
  LEM-02       trial=2 → does_not_entail
  DNE-01       trial=0 → entails
  DNE-01       trial=1 → entails
  DNE-01       trial=2 → entails
  DNE-02       trial=0 → entails
  DNE-02       trial=1 → entails
  DNE-02       trial=2 → entails
  EFQ-01       trial=0 → entails
  EFQ-01       trial=1 → entails
  EFQ-01       trial=2 → entails
  EFQ-02       trial=0 → does_not_entail
  EFQ-02       trial=1 → does_not_entail
  EFQ-02       trial=2 → does_not_entail
  REL-01       trial=0 → entails
  REL-01       trial=1 → entails
  REL-01       trial=2 → entails
  REL-02       trial=0 → entails
  REL-02       trial=1 → entails
  REL-02       trial=2 → entails
  DS-01        trial=0 → entails
  DS-01        trial=1 → entails
  DS-01        trial=2 → entails
  CON-01       trial=0 → does_not_

## Quick sanity check

Per-model verdict consistency (same item, multiple trials):

In [5]:
from collections import defaultdict, Counter

by_model_item = defaultdict(list)
for r in results:
    by_model_item[(r["model"], r["item_id"])].append(r["verdict"])

print("Within-model verdict consistency (1.0 = always same answer):\n")
for (model, item_id), verdicts in sorted(by_model_item.items()):
    most_common = Counter(verdicts).most_common(1)[0]
    consistency = most_common[1] / len(verdicts)
    flag = "⚠" if consistency < 1.0 else " "
    print(f"  {flag} {model:25s} {item_id:12s} {dict(Counter(verdicts))} → consistency={consistency:.2f}")

Within-model verdict consistency (1.0 = always same answer):

    claude-sonnet-4-5         CON-01       {'does_not_entail': 3} → consistency=1.00
    claude-sonnet-4-5         DNE-01       {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         DNE-02       {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         DS-01        {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         EFQ-01       {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         EFQ-02       {'does_not_entail': 3} → consistency=1.00
    claude-sonnet-4-5         LEM-01       {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         LEM-02       {'does_not_entail': 3} → consistency=1.00
    claude-sonnet-4-5         MAT-01       {'entails': 3} → consistency=1.00
  ⚠ claude-sonnet-4-5         PEI-01       {'entails': 2, 'does_not_entail': 1} → consistency=0.67
    claude-sonnet-4-5         REL-01       {'entails': 3} → consistency=1.00
    claude-sonnet-4-5         REL-02       {'

## Next

→ Notebook 11: load `llm_responses.json`, score each model's verdict pattern against each logic's expected pattern. Output: per-model **logic fingerprint** (% match to classical / intuitionistic / paraconsistent / relevance).